# Semaine 7 — Jour 7 : Projet intégré et revue d’architecture

Objectif : Assembler les workflows de réception, traitement et décision.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Pour une étude complète, prévoir 2 à 3 heures avec entretien, challenge et revue. Le premier lancement Docker peut demander un temps supplémentaire.

Prérequis : Python 3.12 recommandé, HTTP/JSON, notions d’agents et de persistance des semaines précédentes. Les commandes Python utilisent uniquement la bibliothèque standard. n8n exige Docker Compose v2 pour le lab d’intégration.

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md) avant d’ouvrir [les corrigés](corriges/exercises_solution.md). Préparer [l’entretien](interview.md) et [le challenge](challenge.md).

Notebooks depuis la racine : `notebooks/week07/S7_J7_projet_integration.ipynb` et `notebooks/week07/teacher/S7_J7_projet_integration_teacher.ipynb`.

Critère de réussite : Présenter une décision d’architecture avec ses limites opérationnelles.

Le lab ne requiert aucune clé LLM. Ses réponses sont simulées et ne mesurent pas la qualité d’un modèle réel.


# Objectifs mesurables

- Assembler les workflows de réception, traitement et décision.
- Démontrer les invariants du parcours par des scénarios de panne.
- Présenter une décision d’architecture avec ses limites opérationnelles.

## Preuves attendues

Une sortie de lab reproductible, une explication de l’invariant étudié et une limite correctement identifiée. Le challenge approfondit le parcours essentiel sans être nécessaire pour finir la session de 60 minutes.


# Semaine 7 — Jour 7 : Projet intégré et revue d’architecture


## 1. Le livrable de la semaine
Vous disposez d’un moteur transactionnel local, d’une API HTTP et de trois workflows n8n. Le parcours complet reçoit une demande, génère un brouillon simulé, attend une décision et écrit une publication locale. Le projet démontre la maîtrise du processus autour d’une capacité IA remplaçable. Il ne publie ni email, ni ticket, ni message dans un service externe.

Le premier workflow transmet un événement. Le deuxième, planifié, appelle tick. Le troisième reçoit une décision explicite et appelle le endpoint de revue. Les workflows utilisent des credentials séparés et ne contiennent aucun secret. Le backend constitue la source d’état métier; les historiques d’exécution n8n servent au diagnostic du transport.

## 2. Contrat de démonstration
Préparer quatre cas : demande nominale approuvée, demande refusée, livraison répétée et demande expirée. Les scripts Python simulent l’horloge pour le cas d’expiration. Le parcours HTTP local démontre les rôles et la réponse 202. L’intégration n8n demande Docker et une configuration des credentials dans l’éditeur; elle possède sa propre checklist.

Le nominal ne suffit pas à prouver la fiabilité. Envoyer un même event_id avec un autre texte doit échouer. Faire échouer le générateur trois fois doit terminer le traitement automatique. Répéter une approbation doit laisser une seule publication. Un mauvais hash ou une identité non autorisée doit être rejeté avant l’effet.

## 3. Une revue doit distinguer preuve et intention
La suite unittest couvre le moteur et l’API en conditions locales. Les notebooks contiennent des exemples déterministes issus de sources Markdown. Les exports n8n sont des fichiers de configuration proposés pour import; sans exécution dans l’éditeur, leur validation reste statique. Le rapport livré indique exactement ce qui a été exécuté.

Ne transformez pas un scénario simulé en preuve de tenue en charge. Le serveur est séquentiel, le tick est borné à cent jobs et la base est SQLite. Un arrêt brutal, une saturation réseau, une corruption de stockage ou des workers concurrents demanderaient des tests supplémentaires. Le projet est un incrément pédagogique qui peut être intégré sans remplacer mini_framework ou ai_platform.

## 4. Note d’architecture
Présenter le problème, les responsabilités, les invariants et les alternatives. Pour ce lab, un backend isolé possède l’état; n8n possède les connecteurs et déclencheurs. Une alternative place plus de logique dans n8n, avec moins de code mais davantage de dépendance à la représentation des nœuds. Une autre utilise un moteur durable dédié, avec un protocole de reprise plus riche et un coût d’exploitation différent.

La décision doit partir des contraintes : nombre de workflows, équipe de maintenance, durée d’attente, besoin de transactions, débit et gouvernance des credentials. Éviter de choisir un outil parce qu’il comporte des capacités agentiques. L’agent est un composant du système, pas une réponse universelle à l’orchestration.

## 5. Passage vers une architecture exploitable
Pour un effet réel, prévoir une outbox et un dispatcher; pour plusieurs workers, un claim atomique et une politique de lease; pour plusieurs relecteurs, une identité individuelle et une autorisation par ressource. Pour le modèle, définir un jeu d’évaluation, les budgets et les garde-fous. Ces éléments sont des propositions d’amélioration documentées, pas des garanties déjà implémentées.

L’exploitation doit distinguer temps de génération, temps d’attente humaine et âge des jobs bloqués. Un SLO de réception peut être respecté alors que les demandes attendent des heures en revue. Les contrôles de la semaine 8 prolongeront cette analyse en traitant l’évaluation, les coûts et la gouvernance. Cette semaine reste centrée sur le workflow.

## À retenir
Un projet intégré est terminé lorsque son parcours et ses échecs attendus sont démontrables, ses fichiers reproductibles et ses limites clairement exposées.


## Diagramme du jour

```mermaid
flowchart TD
  A["Réception n8n"] --> B["Jobs durables"]
  C["Scheduler n8n"] --> D["Génération et contrôles"]
  B --> D
  D --> E["Attente humaine"]
  F["Décision n8n"] --> E
  E --> G["Publication locale ou refus"]
```

Si votre lecteur Jupyter ne rend pas Mermaid, ouvrir le fichier .mmd avec un lecteur compatible.

# Exemple exécutable autonome

Ce fragment illustre le concept du jour. Le moteur complet et ses scénarios sont dans examples/week07.



In [ ]:
states = ["published", "rejected", "expired"]
publications = ["evt-approved"]
assert states.count("published") == len(publications) == 1
assert "rejected" in states and "expired" in states
print({"states": states, "local_publications": len(publications)})


# Lab J7 — Projet intégré et revue d’architecture

Depuis la racine du dépôt :

```bash
python3 book/week07/day07/labs/run.py
```

Résultat attendu : **published=1, rejected=1, expired=1, publication_count=1**.

Le script exécute le scénario du jour contre une base en mémoire et ferme la connexion. Il ne crée aucun message externe. Inspecter `examples/week07/demo.py` et la méthode correspondante dans `workflow.py` pour relier la sortie à la transition étudiée.

## Travail à réaliser

1. Exécuter le scénario et expliquer chaque champ retourné.
2. Réaliser l’exercice 2 du jour avec une entrée différente.
3. Justifier l’absence de publication non autorisée.

Pour le parcours n8n, suivre [le guide d’installation](../../../../docs/week07/QUICKSTART.md). J1 configure la réception; J2 ajoute le scheduler; J3 teste les doublons; J4 observe les reprises et erreurs; J5 étudie le port de génération; J6 importe le workflow de décision; J7 exécute la checklist d’intégration.

Le scénario J4 utilise une horloge virtuelle. Les tests de reprise sont rapides et n’attendent pas réellement les délais. Le projet HTTP utilise une base fichier pour démontrer la persistance après redémarrage.


# Exercices

Faire une première tentative sans consulter les corrigés.

## Exercice 1 — Établir les critères d’acceptation

Proposer cinq contrôles bloquants du parcours complet.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 2 — Lire les compteurs

Trois jobs awaiting_review et aucun published signifient-ils une panne ?

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 3 — Choisir une prochaine évolution

Prioriser une amélioration si le projet devient multi-utilisateur avec effets réels.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.


# Préparation à l’entretien

Répondre en deux minutes par question, avec un exemple concret et une limite.

## Question 1

Comment présenter ce projet en entretien ?

## Question 2

Quand abandonner ce moteur pédagogique ?


# Challenge

Démontrer le parcours complet en cinq minutes, fournir les preuves des branches approuvée, refusée et expirée, puis écrire une ADR courte sur la répartition n8n/backend.

## Critères de revue

Reproductibilité, absence d’effet non autorisé, explication des limites et preuves du comportement attendu.


# Références — J7

Sources consultées le 12 octobre 2026. Les pages évoluent; les exports ciblent la version n8n indiquée dans Compose.

- [Documentation n8n](https://docs.n8n.io/) — vocabulaire et référence des nœuds.
- [Webhook](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.webhook/) — réception et modes de réponse.
- [HTTP Request](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.httprequest/) — appels au backend et credentials.
- [Gestion des erreurs](https://docs.n8n.io/flow-logic/error-handling/) — séparation du flux nominal et des erreurs techniques.
- [Wait](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.wait/) — variante d’attente, non implémentée ici.
- [sqlite3 Python](https://docs.python.org/3/library/sqlite3.html) — transactions et connexions.

Lire en priorité les éléments liés à l’objectif du jour; les autres liens permettent d’approfondir. Les choix du moteur pédagogique sont explicités dans le cours et ne constituent pas une spécification de n8n.


# Corrigé des exercices

## Exercice 1 — Établir les critères d’acceptation

Pas de publication sans approbation; aucun effet sur doublon identique; collision rejetée; validation expirée bloquée; reprise bornée et dead letter observable. Ajouter authentification et persistance selon le périmètre.

## Exercice 2 — Lire les compteurs

Pas nécessairement : les humains peuvent ne pas avoir décidé. Vérifier l’âge, le délai attendu et le nombre d’entrées. Un compteur seul ne distingue pas attente normale et blocage.

## Exercice 3 — Choisir une prochaine évolution

D’abord identité et autorisation individuelles et traitement des effets externes; ensuite concurrence et exploitation selon le débit. Justifier avec les risques et contraintes réels, pas avec une liste d’outils.


# Réponses d’entretien

## Question 1

Expliquer un doublon réseau, le commit avant 202, les gardes de validation, la séparation du LLM et les tests. Donner une limite honnête : la publication est locale, pas un envoi externe garanti unique.

## Question 2

Quand les besoins imposent concurrence, orchestration durable distribuée, isolation multi-client ou exigences d’audit que ce moteur ne fournit pas. Définir les exigences avant la migration.


# Solution du challenge

Exécuter demo 7, la suite unittest, puis les trois workflows n8n si Docker est disponible. Le scénario Python doit produire published:1, rejected:1, expired:1 et une publication. L’ADR expose le contexte, la décision, l’alternative « logique dans n8n », les conséquences et les limites. Une validation n8n non réalisée doit rester marquée non vérifiée.


# Revue pédagogique

## Acquis à vérifier

- Assembler les workflows de réception, traitement et décision.
- Démontrer les invariants du parcours par des scénarios de panne.
- Présenter une décision d’architecture avec ses limites opérationnelles.

## Erreurs fréquentes

Confondre réception et achèvement; assimiler un exemple simulé à une évaluation de modèle; déduire une garantie distribuée d’une transaction locale. Pour ce jour, vérifier en particulier : D’abord identité et autorisation individuelles et traitement des effets externes; ensuite concurrence et exploitation selon le débit. Justifier avec les risques et contraintes réels, pas avec une liste d’outils.

## Barème indicatif sur 10

- 3 points : explication des concepts avec le scénario du support.
- 3 points : reproduction du lab et analyse de sa sortie.
- 2 points : réponses aux exercices, sans confusion sur les garanties.
- 2 points : challenge argumenté et limite opérationnelle identifiée.

Le parcours essentiel peut être validé sans le challenge. Toute publication sans validation bloque la validation du projet intégré, quel que soit le score.


# Correction exécutable du lab

Dans Jupyter, démarrer le kernel depuis la racine du dépôt ou depuis son arbre. Ce fragment recherche la racine contenant le module S7. Il utilise le moteur complet, distinct du petit exemple autonome.



In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples/week07/workflow.py").is_file()), None)
if root is None:
    raise RuntimeError("Ouvrir le notebook depuis le dépôt décompressé")
sys.path.insert(0, str(root))
from examples.week07.demo import scenario
result = scenario(7)
assert result == {'statuses': {'expired': 1, 'published': 1, 'rejected': 1}, 'publication_count': 1}
print(result)



Les assertions constituent la preuve minimale de ce scénario; les tests de `tests/week07` couvrent les branches complémentaires. Exécuter demo 7, la suite unittest, puis les trois workflows n8n si Docker est disponible. Le scénario Python doit produire published:1, rejected:1, expired:1 et une publication. L’ADR expose le contexte, la décision, l’alternative « logique dans n8n », les conséquences et les limites. Une validation n8n non réalisée doit rester marquée non vérifiée.
